### LAB 03: Word Representation and Embeddings

In [2]:
%load_ext autoreload
%autoreload 2
import numpy as np
import pandas as pd
from cooccurrence import load_documents, split_sen, build_vocabulary, build_cooccurrence_matrix, cosine_similarity, most_similar, tokenize
from gensim.models import Word2Vec
import time 
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter

#### 10. Experiment 1 - Word - context representation

In [3]:
docs = load_documents(10000)
sents = split_sen(docs)
vocab = build_vocabulary(sents)
print(f'{len(docs):,} documents | {len(sents):,} sentences | {sum(len(s) for s in sents):,} tokens | |V| = {len(vocab):,}')


10,000 documents | 214,053 sentences | 3,624,214 tokens | |V| = 100,986


In [24]:
pairs = [("doctor", "physician"), ("doctor", "hospital"), ("car", "automobile"), ("king", "queen"), ("cat", "dog"), ("computer", "banana")]
n = len(vocab)

res_window = {}
matrixes = {}

for window in [1, 2, 5]:
    X = build_cooccurrence_matrix(sents, vocab, window=window)
    matrixes[window] = X

    res = {"Vocab size":n , 'Matrix size': n*n , "Number of non-zero values": X.nnz, "Sparsity": 100 - (X.nnz/(n*n))*100}

    for x,y, in pairs:
        res[f'{x} - {y}'] = cosine_similarity(X[vocab[x]].toarray().ravel(), X[vocab[y]].toarray().ravel())
    res_window[f'Window = {window}'] = res

pd.DataFrame(res_window).round(2).T

,Vocab size,Matrix size,Number of non-zero values,Sparsity,doctor - physician,doctor - hospital,car - automobile,king - queen,cat - dog,computer - banana
Window = 1,100986.0,1.019817e+10,2188113.0,99.98,0.82,0.65,0.35,0.67,0.71,0.40
Window = 2,100986.0,1.019817e+10,4193446.0,99.96,0.81,0.75,0.52,0.79,0.80,0.56
Window = 5,100986.0,1.019817e+10,8789383.0,99.91,0.89,0.82,0.69,0.89,0.87,0.73


Top-5 similarity

In [25]:
test_word = ["doctor","hospital","patient","disease","computer","football","banana"]

for word in test_word:
    print(f"Most similar words to '{word}':")
    table = {}

    for w, X in matrixes.items():

        result = most_similar(word, X, vocab, top_k=5)
        table[f"Window = {w}"] = [
            f"{similar_word} ({similarity:.4f})"
            for similar_word, similarity in result
        ]

    df = pd.DataFrame(table, index=range(1, 6))
    display(df)

Most similar words to 'doctor':


,Window = 1,Window = 2,Window = 5
1,child (0.9089),child (0.9078),child (0.9313)
2,computer (0.8910),business (0.8812),schedule (0.9272)
3,chinchilla (0.8824),vehicle (0.8726),plan (0.9239)
4,business (0.8792),dog (0.8697),job (0.9223)
5,dentist (0.8708),computer (0.8660),business (0.9149)


Most similar words to 'hospital':


,Window = 1,Window = 2,Window = 5
1,hotel (0.9018),position (0.9139),house (0.9367)
2,game (0.8924),hotel (0.9095),community (0.9330)
3,competition (0.8875),house (0.9085),location (0.9320)
4,market (0.8857),place (0.9079),office (0.9296)
5,church (0.8849),building (0.9013),present (0.9289)


Most similar words to 'patient':


...

,Window = 1,Window = 2,Window = 5
1,business (0.9134),business (0.9105),business (0.9466)
2,doctor (0.8910),budget (0.8802),home (0.9352)
3,child (0.8768),home (0.8705),print (0.9256)
4,chinchilla (0.8728),doctor (0.8660),vehicle (0.9226)
5,vehicle (0.8493),vehicle (0.8644),budget (0.9219)


Most similar words to 'football':


,Window = 1,Window = 2,Window = 5
1,yaozu (0.5704),terran (0.5481),shirt (0.8446)
2,tee (0.5465),tee (0.5363),replica (0.8286)
3,extolling (0.4364),shirt (0.4750),kits (0.8279)
4,cuca (0.4364),shirtn (0.4059),yaozu (0.7688)
5,terran (0.4117),shirtm (0.4010),terran (0.7642)


Most similar words to 'banana':


,Window = 1,Window = 2,Window = 5
1,word (0.6710),blue (0.7320),running (0.8167)
2,inside (0.6637),race (0.7191),table (0.8157)
3,future (0.6600),light (0.7185),for (0.8156)
4,moment (0.6577),military (0.7184),light (0.8151)
5,region (0.6543),design (0.7181),bar (0.8140)


In [26]:
rows = [("exp1", setting, metric, value) for setting, res in res_window.items() for metric, value in res.items()]
pd.DataFrame(rows, columns=["experiment", "setting", "metric", "value"]).to_csv("results.csv", index=False)


#### 14–15. CBOW và Skip-gram
**CBOW và Skip-gram khác nhau ở input/target như thế nào?**

|         | CBOW                | Skip-gram           |
| ------- | ------------------- | ------------------- |
| Input   | Các từ xung quanh (context) | Từ trung tâm (target) |
| Output  | Từ trung tâm        | Từng từ xung quanh  |
| Ví dụ   | `the, eats → cat`   | `cat → the`, `cat → eats` |
| Tốc độ  | Thường nhanh hơn    | Thường chậm hơn     |
| Từ hiếm | Thường kém hơn      | Thường tốt hơn      |
| Ý tưởng | Context → word      | Word → context      |

#### 16. Bài tập prediction - CBOW vs Skip-gram

Cho câu: `the cat eats fish` với window = 1 → context của mỗi từ là từ ngay trước và ngay sau nó.

**CBOW** (context → target): 4 training examples

| # | Input (context) | Target |
|---|---|---|
| 1 | [cat] | the |
| 2 | [the, eats] | cat |
| 3 | [cat, fish] | eats |
| 4 | [eats] | fish |

**Skip-gram** (target → context): 6 training pairs

| # | Input (target) | Output (context) |
|---|---|---|
| 1 | the | cat |
| 2 | cat | the |
| 3 | cat | eats |
| 4 | eats | cat |
| 5 | eats | fish |
| 6 | fish | eats |

Cùng một câu, Skip-gram sinh nhiều mẫu hơn CBOW (6 so với 4) vì mỗi cặp (target, context) là một mẫu riêng.

#### 17. Experiment 2 - Word2Vec

In [27]:
model = Word2Vec(sentences=sents, vector_size=100, window=5, min_count=5, sg=1, negative=5, epochs=10,
                 workers=4, seed=42)

print(f'|V| = {len(model.wv):,} | {model.corpus_total_words:,} tokens')

|V| = 28,956 | 3,624,214 tokens


#### 18. Inspect embeddings

In [28]:
words = ["doctor", "hospital", "patient", "disease", "computer", "football", "banana"]
top5 = {w: [f"{x} ({s:.4f})" for x, s in model.wv.most_similar(w, topn=5)] for w in words}
pd.DataFrame(top5, index=range(1, 6))

,doctor,hospital,patient,disease,computer,football,banana
1,surgeon (0.7205),icu (0.6748),patient's (0.6899),infection (0.7600),computers (0.6811),replica (0.8854),sprouts (0.7059)
2,dentist (0.7108),navicent (0.6455),patients (0.6466),chronic (0.7599),desktop (0.6450),shirt (0.8323),beef (0.7035)
3,ophthalmologist (0.6959),airlifted (0.6381),hospice (0.6446),autoimmune (0.7503),laptop (0.6442),shirts (0.7975),smoothie (0.6970)
4,physician (0.6891),maternal (0.6350),diagnosis (0.6260),hypertension (0.7481),defraggler (0.6429),thcheap (0.7336),almond (0.6952)
5,veterinarian (0.6745),johns (0.6333),dialysis (0.6104),artery (0.7464),phone's (0.6266),shirtr (0.6803),chilli (0.6902)


In [29]:
def evidence(a, b, n=3):
    hits = [" ".join(s) for s in sents if a in s and b in s]
    print(f"'{a}' va '{b}' cung xuat hien trong {len(hits)} cau")
    for h in hits[:n]:
        print("   -", h[:200])


evidence("doctor", "surgeon")
evidence("football", model.wv.most_similar("football", topn=1)[0][0])


'doctor' va 'surgeon' cung xuat hien trong 6 cau
   - when you're considering lasik you will want to speak to your present eye doctor to learn if you're an excellent candidate and receive a referral to an eye surgeon
   - you should make certain the surgeon is well qualified if you would like to get the ideal eye doctor
   - if you are searching for a lasik surgeon the very best eye doctor is a person who has performed several thousand eye operations already
'football' va 'replica' cung xuat hien trong 69 cau
   - i know that all replica football shirt demons don t france soccer jersey long sleeve really believe that thcheap football shirts cheap football shirts true
   - thcheap football shirts baby was personally experimented by replica football shirtir elders
   - football shirts old black had been with ye hao ac milan white jersey for so long and replica football shirt mouth had already been honed very hard


#### 19. Experiment 3: Context window

In [30]:
def train(window=5, vector_size=100, seed=42):
    start = time.time()
    m = Word2Vec(sentences=sents, vector_size=vector_size, window=window, min_count=5,
                 sg=1, negative=5, epochs=10, workers=4, seed=seed)
    return m, time.time() - start


def save(exp, table):
    old = pd.read_csv("results.csv")
    old = old[old.experiment != exp]
    new = table.stack().reset_index()
    new.columns = ["setting", "metric", "value"]
    new.insert(0, "experiment", exp)
    pd.concat([old, new]).to_csv("results.csv", index=False)


In [31]:
pairs = [("doctor", "physician"), ("doctor", "hospital"), ("cat", "dog")]
rows = []

for wd in [2,5,10]:
    for seed in [42, 123, 456]:
        m, t = train(window=wd, seed=seed)
        row = {"window": wd, "seed": seed, "training time": t}
        for a, b in pairs:
            rows.append({"Word pair": f"{a}-{b}", "window": wd, "seed": seed,
                         "similarity": float(m.wv.similarity(a, b))})
        
        print(f"window={wd} seed={seed}({t:.0f}s)")

exp3 = pd.DataFrame(rows)

window=2 seed=42(18s)
window=2 seed=123(19s)
window=2 seed=456(18s)
window=5 seed=42(29s)
window=5 seed=123(28s)
window=5 seed=456(33s)
window=10 seed=42(47s)
window=10 seed=123(45s)
window=10 seed=456(43s)


In [32]:
mean = exp3.pivot_table(index="Word pair", columns="window", values="similarity", aggfunc="mean")
std = exp3.pivot_table(index="Word pair", columns="window", values="similarity", aggfunc="std")

table = mean.round(3).astype(str) 
table.columns = [f"Window {w}" for w in table.columns]
table = table.loc[[f"{a}-{b}" for a, b in pairs]]

save("exp3", mean.rename(columns=lambda w: f"window={w}").T.round(4))
table


,Window 2,Window 5,Window 10
Word pair,,,
doctor-physician,0.711,0.701,0.693
doctor-hospital,0.537,0.545,0.535
cat-dog,0.566,0.518,0.569


**Window lớn hơn có luôn tốt hơn không?**

**Không.**

- **Similarity không tăng theo window.** Chỉ doctor-physician giảm đều nhẹ (0.711 → 0.701 → 0.693); doctor-hospital cao nhất ở window 5, cat-dog lại thấp nhất ở window 5. Không có cặp nào tăng đều khi window tăng.
- **Chi phí tăng rõ rệt:** window 10 train lâu gấp khoảng 2.5 lần window 2, vì mỗi từ sinh ra nhiều cặp (target, context) hơn để học.



#### 20. Experiment 4: Embeddings dimension

In [33]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from gensim.test.utils import datapath
from cooccurrence import tokenize

imdb = pd.read_csv("IMDB Dataset.csv").sample(10000, random_state=42)
reviews = [tokenize(r.replace("<br />", " ")) for r in imdb["review"]]
labels = (imdb["sentiment"] == "positive").astype(int).values

train_tokens, test_tokens, y_train, y_test = train_test_split(
    reviews, labels, test_size=0.2, random_state=42, stratify=labels)
print(len(train_tokens), "review train,", len(test_tokens), "review test")


def doc_vector(tokens, wv):
    vecs = [wv[w] for w in tokens if w in wv]
    if not vecs:
        return np.zeros(wv.vector_size)
    return np.mean(vecs, axis=0)


def downstream_accuracy(wv):
    X_train = [doc_vector(t, wv) for t in train_tokens]
    X_test = [doc_vector(t, wv) for t in test_tokens]
    clf = LogisticRegression(max_iter=1000).fit(X_train, y_train)
    return clf.score(X_test, y_test)


8000 review train, 2000 review test


In [34]:
rows = []
for dim in [50, 100, 300]:
    for seed in [42, 123, 456]:
        m, t = train(vector_size=dim, seed=seed)
        row = {
            "dim": dim,
            "seed": seed,
            "Training time (s)": t,
            "Model size (MB)": m.wv.vectors.nbytes / 2**20,
            "Analogy accuracy": m.wv.evaluate_word_analogies(datapath("questions-words.txt"))[0],
            "Downstream accuracy": downstream_accuracy(m.wv),
        }
        for a, b in pairs:
            row[f"{a}-{b}"] = float(m.wv.similarity(a, b))
        rows.append(row)
        print(f"dim={dim} seed={seed} xong ({t:.0f}s)")

exp4 = pd.DataFrame(rows)


dim=50 seed=42 xong (27s)
dim=50 seed=123 xong (27s)
dim=50 seed=456 xong (27s)
dim=100 seed=42 xong (32s)
dim=100 seed=123 xong (30s)
dim=100 seed=456 xong (30s)
dim=300 seed=42 xong (52s)
dim=300 seed=123 xong (51s)
dim=300 seed=456 xong (51s)


In [35]:
mean = exp4.drop(columns="seed").groupby("dim").mean()
std = exp4.drop(columns="seed").groupby("dim").std()

table = (mean.round(3).astype(str) + " ± " + std.round(3).astype(str)).T
table.columns = [f"Dim {d}" for d in table.columns]

save("exp4", mean.rename(index=lambda d: f"dim={d}").round(4))
table


,Dim 50,Dim 100,Dim 300
Training time (s),26.984 ± 0.132,30.297 ± 1.305,51.081 ± 0.764
Model size (MB),5.523 ± 0.0,11.046 ± 0.0,33.138 ± 0.0
Analogy accuracy,0.108 ± 0.006,0.12 ± 0.003,0.098 ± 0.002
Downstream accuracy,0.709 ± 0.004,0.729 ± 0.002,0.756 ± 0.002
doctor-physician,0.855 ± 0.008,0.702 ± 0.021,0.553 ± 0.008
doctor-hospital,0.636 ± 0.011,0.526 ± 0.035,0.413 ± 0.016
cat-dog,0.692 ± 0.022,0.565 ± 0.053,0.444 ± 0.006


**Nhận xét Experiment 4 — trade-off dimension ↔ capacity ↔ data ↔ computation:**
- **Model size** tăng tuyến tính theo dimension: 5.5 → 11.0 → 33.1 MB (|V| × dim × 4 byte).
- **Training time:** 27s → 30s → 51s — từ 50 lên 100 gần như không đổi, lên 300 chậm hơn khoảng 1.7 lần.
- **Downstream (IMDB sentiment):** 0.709 → 0.729 → 0.756, độ lệch giữa các seed ≤ 0.004 → cải thiện thật: nhiều chiều hơn giữ được nhiều thông tin hơn cho bài toán phân loại.
- **Analogy accuracy:** 0.108 → **0.120** → 0.098 — đạt đỉnh ở dim 100 rồi giảm ở dim 300: corpus 3.6 triệu token không đủ dữ liệu để học tốt 300 chiều (capacity vượt quá data).
- **Cosine giảm khi dim tăng** (doctor–physician 0.855 → 0.702 → 0.553): không có nghĩa model kém đi — vector càng nhiều chiều thì cosine nói chung càng nhỏ, nên không so sánh trực tiếp cosine giữa các dim; phải dựa vào analogy/downstream.
- **Kết luận:** dim 100 là điểm cân bằng với lượng dữ liệu này; dim 300 chỉ đáng dùng khi có nhiều dữ liệu hơn hoặc cần downstream tốt nhất và chấp nhận chi phí gấp 3 bộ nhớ.

#### 21. Evaluation - Word similarity

In [36]:
sim_pairs = [("doctor", "physician"), ("car", "automobile"), ("king", "queen"),
             ("cat", "dog"), ("computer", "banana")]
my_rank = [1, 2, 3, 4, 5]

sim_table = pd.DataFrame(sim_pairs, columns=["word 1", "word 2"])
sim_table["similarity"] = [round(float(model.wv.similarity(a, b)), 3) for a, b in sim_pairs]
sim_table["model rank"] = sim_table["similarity"].rank(ascending=False).astype(int)
sim_table["my rank"] = my_rank
sim_table.sort_values("model rank")


,word 1,word 2,similarity,model rank,my rank
0,doctor,physician,0.689,1,1
1,car,automobile,0.612,2,2
3,cat,dog,0.579,3,4
2,king,queen,0.567,4,3
4,computer,banana,0.049,5,5


In [ ]:
from collections import Counter


def next_words(word, n=8):
    cnt = Counter(s[i + 1] for s in sents for i, w in enumerate(s[:-1]) if w == word)
    return cnt.most_common(n)


print("tu dung sau 'king': ", next_words("king"))
print("tu dung sau 'queen':", next_words("queen"))
bed = sum(1 for s in sents for i, w in enumerate(s[:-1])
          if w in ("king", "queen") and s[i + 1] in ("size", "bed", "beds", "mattress"))
print("so lan king/queen di voi size/bed/mattress:", bed)

**Nhận xét Word similarity:**
1. **Xếp hạng (model):** doctor–physician (0.689) > car–automobile (0.612) > cat–dog (0.579) > king–queen (0.567) > computer–banana (0.049).
2. **So với trực giác:** khớp ở hạng 1, 2 và 5; computer–banana gần 0 đúng như kỳ vọng. Lệch ở hạng 3–4: cat–dog vượt king–queen.
3. **Trường hợp bất ngờ — king–queen thấp hơn cat–dog:** trong corpus, king và queen thường **không** mang nghĩa hoàng gia: từ đứng sau "king" hay gặp là *size* (18 lần), *comforter*, *county*, *george*; sau "queen" là *size* (22), *bee* (14), *elizabeth*, *comforter*. Có 46 lần king/queen đi với size/bed/mattress (cỡ giường) → vector của hai từ bị pha trộn nhiều nghĩa (polysemy). Kết quả analogy cũng lộ điều này: king − man + woman ra queen nhưng tiếp theo là *robes, sophia, bed, duchess*.

#### 22. Word analogy

In [37]:
for pos, neg in [(["king", "woman"], ["man"]), (["paris", "italy"], ["france"])]:
    res = model.wv.most_similar(positive=pos, negative=neg, topn=5)
    print(f"{pos[0]} - {neg[0]} + {pos[1]} =", [f"{w} ({s:.2f})" for w, s in res])

wv = model.wv
print("\ncos(woman - man, queen - king) =", round(cosine_similarity(wv["woman"] - wv["man"], wv["queen"] - wv["king"]), 3))
print("cos(italy - france, rome - paris) =", round(cosine_similarity(wv["italy"] - wv["france"], wv["rome"] - wv["paris"]), 3))


king - man + woman = ['queen (0.52)', 'robes (0.50)', 'sophia (0.48)', 'bed (0.46)', 'duchess (0.46)']
paris - france + italy = ['vilnius (0.58)', 'loughrea (0.57)', 'venice (0.56)', 'anniversary (0.55)', 'tuscany (0.55)']

cos(woman - man, queen - king) = 0.203
cos(italy - france, rome - paris) = 0.144


**Tại sao vector arithmetic có thể biểu hiện một số quan hệ?**

Vector của một từ được học từ ngữ cảnh của nó, nên hiệu hai vector ghi lại **sự khác nhau về ngữ cảnh** của hai từ. Nếu một quan hệ (nam ↔ nữ, nước ↔ thủ đô) làm ngữ cảnh thay đổi **một cách nhất quán** trong corpus (ví dụ man/king hay đi với *he, his*, woman/queen hay đi với *she, her*), thì hiệu vector của các cặp có cùng quan hệ gần như **song song**: woman − man ≈ queen − king. Khi đó king − man + woman ≈ king + (queen − king) = queen — cộng trừ vector trở thành "di chuyển theo quan hệ".



#### 23. Bài tập tính analogy
Cho các vector giả định: king = [8, 2, 7], man = [5, 1, 5], woman = [5, 3, 5]

Tính: king − man + woman

- king − man = [8, 2, 7] − [5, 1, 5] = [3, 1, 2]
- king − man + woman = [3, 1, 2] + [5, 3, 5] = **[8, 4, 7]**

**Vector mới có thể đại diện cho quan hệ nào?**

woman − man = [0, 2, 0] → chỉ chiều thứ 2 khác nhau, nên chiều 2 mang tín hiệu **giới tính**. Vector [8, 4, 7] giữ nguyên chiều 1 và 3 của king (các đặc trưng "hoàng gia") và chỉ đổi chiều giới tính → đó là "king nhưng là nữ", tức biểu diễn quan hệ **nam ↔ nữ** (man : woman = king : queen). Từ phù hợp nhất ở vị trí này là **queen**.

#### 24. Semantic search

In [38]:
doc_tokens = [tokenize(d) for d in docs]
doc_vecs = np.array([doc_vector(t, model.wv) for t in doc_tokens])

tfidf = TfidfVectorizer(tokenizer=tokenize, lowercase=False, token_pattern=None)
doc_tfidf = tfidf.fit_transform(docs)


def expand_query(query, k=3):
    words = [w for w in tokenize(query) if w in model.wv]
    extra = [x for w in words for x, _ in model.wv.most_similar(w, topn=k)]
    return words + extra


def semantic_search(query, top=5):
    q = doc_vector(expand_query(query), model.wv)
    scores = np.array([cosine_similarity(q, v) for v in doc_vecs])
    return np.argsort(-scores)[:top], scores


def tfidf_search(query, top=5):
    scores = (doc_tfidf @ tfidf.transform([query]).T).toarray().ravel()
    return np.argsort(-scores)[:top], scores


In [39]:
for query in ["medical treatment", "cheap flight"]:
    print(f"\n=== {query} -> {expand_query(query)}")
    query_words = set(tokenize(query))
    for name, search in [("Word2Vec", semantic_search), ("TF-IDF", tfidf_search)]:
        print(f"  {name}:")
        idx, scores = search(query)
        for i in idx:
            mark = "*" if not query_words & set(doc_tokens[i]) else " "
            print(f"    {scores[i]:.3f} {mark} {' '.join(docs[i].split())[:90]}")



=== medical treatment -> ['medical', 'treatment', 'physician', 'veterinary', 'malpractice', 'surgical', 'glaucoma', 'cataract']
  Word2Vec:
    0.843 * A phase II pilot study to evaluate use of intravenous lidocaine for opioid-refractory pain
    0.836   Treatment of bleeding problems without removal of uterus (TCRE) Etc. Screening and treatme
    0.818   Dr. Slatore is a Pulmonary/Critical Care physician and a Core Investigator at the VA Portl
    0.812   Specific focus on the use of emerging technology for the treatment of patients and fabrica
    0.807   Hematology-oncology: The diagnosis, treatment and prevention of blood diseases (hematology
  TF-IDF:
    0.453   What is a Online Medical Second Opinion? For over 25 years, patients and families have loo
    0.306   Undergraduate and mecical graduate program tuition for students who. Apr 12, 2018. So dive
    0.301   This title is a comprehensive account of the key aspects of medical leadership. A highly a
    0.296   Based on dema

**Nhận xét Semantic search vs TF-IDF (LAB 01):**
- **medical treatment:** query được mở rộng thành physician, veterinary, malpractice, surgical, glaucoma, cataract. Kết quả top 1 của Word2Vec là bài nghiên cứu về lidocaine (dấu `*`) — **không chứa chữ medical hay treatment** nhưng đúng chủ đề y khoa; TF-IDF không bao giờ tìm được bài này vì điểm bằng 0. Trong khi đó TF-IDF xếp cao một bài về học phí ("mecical graduate program tuition") chỉ vì khớp chữ "medical".
- **cheap flight:** từ mở rộng có **nhiễu** — "cannula" (ống thở oxy) vì nhiều trang quảng cáo "cheap oxygen nasal cannula"; nhưng phần lớn kết quả vẫn đúng chủ đề vé máy bay (Qantas, airfares, airport car rental). TF-IDF lại trả về bài "antisubmarine warfare" do chữ "flight" theo nghĩa quân sự (polysemy).
- **Kết luận:** semantic search tìm được document liên quan **không chứa từ của query**, điều TF-IDF không làm được; nhưng chất lượng phụ thuộc vào embedding (từ mở rộng nhiễu → kết quả nhiễu) và việc lấy trung bình vector làm mất thứ tự từ. Đây là bước chuẩn bị cho sentence/document embeddings → semantic retrieval → RAG.

#### 25. Error analysis

In [50]:
from collections import Counter

freq = Counter(w for s in sents for w in s)
common = {w for w, _ in freq.most_common(100)}
doc_freq = Counter(w for t in doc_tokens for w in set(t))
context_cache = {}


def top_contexts(word, n=8, window=5):
    if word not in context_cache:
        cnt = Counter()
        for s in sents:
            for i, w in enumerate(s):
                if w == word:
                    cnt.update(x for x in s[max(0, i - window):i] + s[i + 1:i + window + 1] if x not in common)
        context_cache[word] = [w for w, _ in cnt.most_common(20)]
    return context_cache[word][:n]


def analyze(a, b):
    ca, cb = top_contexts(a, n=20), top_contexts(b, n=20)
    hits = [" ".join(s) for s in sents if a in s and b in s]
    print(f"{a} -> {b}: similarity = {model.wv.similarity(a, b):.3f}")
    print(f"   tan suat: {a} = {freq[a]}, {b} = {freq[b]} (xuat hien trong {doc_freq[b]} documents)")
    print(f"   {a} hay di cung: {', '.join(ca[:8])}")
    print(f"   {b} hay di cung: {', '.join(cb[:8])}")
    print(f"   ngu canh chung: {', '.join(x for x in ca if x in cb) or '(khong co)'}")
    print(f"   so cau chua ca hai: {len(hits)}")
    for h in hits[:1]:
        print("      -", h[:160])
    print()


cases = {
    "A1. doctor -> surgeon, dentist, physician (nghe y)":
        [("doctor", "surgeon"), ("doctor", "dentist"), ("doctor", "physician")],
    "A2. disease -> infection, chronic, autoimmune (chu de benh)":
        [("disease", "infection"), ("disease", "chronic"), ("disease", "autoimmune")],
    "A3. computer -> desktop, laptop (thiet bi)":
        [("computer", "desktop"), ("computer", "laptop")],
    "B1. football -> replica, shirt (spam ban ao)":
        [("football", "replica"), ("football", "shirt")],
    "B2. banana -> sprouts, smoothie, chilli, almond (tu hiem)":
        [("banana", "sprouts"), ("banana", "smoothie"), ("banana", "chilli"), ("banana", "almond")],
    "B3. patient -> patient's, patients (bien the cua cung mot tu)":
        [("patient", "patient's"), ("patient", "patients")],
}

for title, pairs in cases.items():
    print("=" * 90)
    print(title)
    print("=" * 90)
    for a, b in pairs:
        analyze(a, b)

print("=" * 90)
print("Bang chung bo sung")
print("=" * 90)
football_docs = [t for t in doc_tokens if "football" in t]
spam_docs = [t for t in football_docs if "replica" in t or "thcheap" in t]
total = sum(t.count("football") for t in football_docs)
spam = sum(t.count("football") for t in spam_docs)
print(f"B1: football co mat trong {len(football_docs)} documents, nhung chi {len(spam_docs)} documents spam "
      f"da chiem {spam}/{total} lan xuat hien ({spam / total:.0%})")
print("B2: ", end="")
evidence("banana", "peppers", n=1)


A1. doctor -> surgeon, dentist, physician (nghe y)
doctor -> surgeon: similarity = 0.720
   tan suat: doctor = 263, surgeon = 117 (xuat hien trong 24 documents)
   doctor hay di cung: eye, should, talk, care, health, consult, order, go
   surgeon hay di cung: lasik, eye, you're, should, surgery, plastic, ask, best
   ngu canh chung: eye, should, ask, lasik
   so cau chua ca hai: 6
      - when you're considering lasik you will want to speak to your present eye doctor to learn if you're an excellent candidate and receive a referral to an eye surge

doctor -> dentist: similarity = 0.711
   tan suat: doctor = 263, dentist = 49 (xuat hien trong 22 documents)
   doctor hay di cung: eye, should, talk, care, health, consult, order, go
   dentist hay di cung: visit, cavities, important, see, weatherford, you're, pregnant, dental
   ngu canh chung: go, visit
   so cau chua ca hai: 1
      - talk with your doctor and dentist before the visit about the best way to take care of your blood glucose 

#### 26. Polysemy

In [41]:
print("bank ->", [w for w, _ in model.wv.most_similar("bank", topn=10)])
print(f"sim(bank, money) = {model.wv.similarity('bank', 'money'):.3f}")
print(f"sim(bank, river) = {model.wv.similarity('bank', 'river'):.3f}")
print("context:", top_contexts("bank"))
evidence("bank", "money")
evidence("bank", "river")


bank -> ['phoebus', 'credit', 'citibank', 'debit', 'securities', 'merrill', 'ifsc', 'ammadam', 'hsbc', 'banks']
sim(bank, money) = 0.398
sim(bank, river) = 0.186
context: ['account', 'bank', 'america', 'credit', 'reserve', 'national', 'world', 'central', 'card', 'indian']
'bank' va 'money' cung xuat hien trong 21 cau
   - for example without a bank how would people send money to each other
   - in the libor scandal of mid 2012 accusations were made that libor had been manipulated by the banks that determine the benchmark's rate allegedly since the early 90s to either make money or to make th
   - het beste online casino vinden het beste wwe money in the bank 2019 ergebnisse casino is voor iedereen anders
'bank' va 'river' cung xuat hien trong 2 cau
   - the pass in the 1800's with its designer thomas bain photo swartberg mountain toursonce over the river the road bends to the right and follows the eastern bank of the river
   - the iwrm planning guidelines as recently developed for the

**Làm thế nào một vector duy nhất có thể đại diện cho hai nghĩa?**

**Không thể đại diện tốt.** Word2Vec học một vector cho mỗi từ từ tất cả ngữ cảnh của nó, nên vector của "bank" là một kiểu trung bình có trọng số theo tần suất của các nghĩa:
- Top-10 của bank toàn nghĩa tài chính (credit, citibank, debit, securities, hsbc…); sim(bank, money) = 0.398 so với sim(bank, river) = 0.186.
- Trong corpus: 21 câu có bank và money, chỉ 2 câu có bank và river — và một trong hai câu đó lại là "world bank" (nghĩa tài chính). Nghĩa "bờ sông" chỉ xuất hiện khoảng 1 lần trong 214,053 câu.
- Kết quả: nghĩa phổ biến lấn át nghĩa hiếm. Câu "We sat on the river bank" vẫn nhận vector mang nghĩa ngân hàng.


In [ ]:
X = matrixes[5]
row = X[vocab["bank"]]
print(f"Co-occurrence: vector 'bank' co {X.shape[1]:,} chieu, chi {row.nnz:,} so khac 0 ({row.nnz / X.shape[1]:.2%})")

d = doc_tfidf[0]
print(f"TF-IDF: vector document 0 co {d.shape[1]:,} chieu, chi {d.nnz:,} so khac 0 ({d.nnz / d.shape[1]:.2%})")

v = model.wv["bank"]
print(f"Word2Vec: vector 'bank' co {len(v)} chieu, {np.count_nonzero(v)} so khac 0")

s1 = tokenize("I deposited money in the bank.")
s2 = tokenize("We sat on the river bank.")
print("Word2Vec: 'bank' trong 2 cau co cung mot vector?", np.array_equal(model.wv[s1[-1]], model.wv[s2[-1]]))